# DPO fallback on Colab Pro (L4 preferred, T4 fallback)

One command per docs/DESIGN.md: clone → install → secrets → `scripts/dpo_colab.py`.

That bootstrap downloads the tokenizer + preference pairs (issue 04's artifact, the *same*
pairs the Reward Model consumes) from the Hub, then runs the DPO stage (ts2-dpo with
--resume) — which initializes both the policy and the frozen reference from the SFT
checkpoint via `[init].hub_source`, trains with the hand-written DPO loss, and checkpoints
back to the Hub. It records the held-out reward margin in the artifact's manifest.json. It
is idempotent: after an L4 preemption, just re-run the last cell to resume from the latest
Hub checkpoint. The output checkpoint is a drop-in third model for the eval suite (issue
07). All logic lives in the package/script; edit `configs/dpo_full.toml`, not here. See
`docs/colab-notes.md` for the run procedure.

Before running: set `HF_TOKEN` and `WANDB_API_KEY` in Colab **Secrets** (key icon, left sidebar),
set the repo URL below, and on a T4 change `precision = "fp16"` in the config (Turing has no bf16).

In [ ]:
REPO_URL = "https://github.com/harryct229/tinystories_v2.git"
!git clone {REPO_URL}
%cd tinystories_v2
!pip install -q -e '.[track]'

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

In [ ]:
!python scripts/dpo_colab.py